In [38]:
#importing the frameworks:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [39]:

#taking a look at the data:
df = pd.read_csv("/home/bilalerremach/projects/netflix-content-trends/data/netflix_titles.csv")
print(df.head())

  show_id     type                  title         director  \
0      s1    Movie   Dick Johnson Is Dead  Kirsten Johnson   
1      s2  TV Show          Blood & Water              NaN   
2      s3  TV Show              Ganglands  Julien Leclercq   
3      s4  TV Show  Jailbirds New Orleans              NaN   
4      s5  TV Show           Kota Factory              NaN   

                                                cast        country  \
0                                                NaN  United States   
1  Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...   South Africa   
2  Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...            NaN   
3                                                NaN            NaN   
4  Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...          India   

           date_added  release_year rating   duration  \
0  September 25, 2021          2020  PG-13     90 min   
1  September 24, 2021          2021  TV-MA  2 Seasons   
2  September 24, 2021        

In [40]:
#informations about the data:
print(df.info())

#the messy columns
print(df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB
None
show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64


In [41]:
#data cleanig...

#removing spaces from date_added colum,n
df['date_added'].str.strip()

#convert date_added to datetime
df['date_added'] = pd.to_datetime(df['date_added'], format='mixed')

#extract the year_added
df['year_added'] = df['date_added'].dt.year.astype('Int64')
print(df[['date_added', 'year_added']].head(10))

  date_added  year_added
0 2021-09-25        2021
1 2021-09-24        2021
2 2021-09-24        2021
3 2021-09-24        2021
4 2021-09-24        2021
5 2021-09-24        2021
6 2021-09-24        2021
7 2021-09-24        2021
8 2021-09-24        2021
9 2021-09-24        2021


In [42]:
#filling the empty columns with "unkown"
df['country'] = df['country'].fillna("Unkown")
df['director'] = df['director'].fillna("Unkown")
df['cast'] = df['cast'].fillna("Unkown")
print(df.isna().sum())

show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added      10
release_year     0
rating           4
duration         3
listed_in        0
description      0
year_added      10
dtype: int64


In [43]:
#split the duration into number and unit
df[['duration_val', 'duration_unit']] = df['duration'].str.split(' ', n=1, expand=True)
print(df[['duration_val', 'duration_unit']].head(10))

  duration_val duration_unit
0           90           min
1            2       Seasons
2            1        Season
3            1        Season
4            2       Seasons
5            1        Season
6           91           min
7          125           min
8            9       Seasons
9          104           min


In [44]:
#checking the duplicates
print(df.duplicated().sum())

0


In [45]:
#adding a release_lag column
df['release_lag'] = df['year_added'] - df['release_year']
print(df[['title', 'year_added', 'release_year', 'release_lag']].head(10))

                              title  year_added  release_year  release_lag
0              Dick Johnson Is Dead        2021          2020            1
1                     Blood & Water        2021          2021            0
2                         Ganglands        2021          2021            0
3             Jailbirds New Orleans        2021          2021            0
4                      Kota Factory        2021          2021            0
5                     Midnight Mass        2021          2021            0
6  My Little Pony: A New Generation        2021          2021            0
7                           Sankofa        2021          1993           28
8     The Great British Baking Show        2021          2021            0
9                      The Starling        2021          2021            0


In [46]:
#converting the country and the listed_in into lists
df['country'] = df['country'].str.split(', ')
df = df.explode('country', ignore_index=True)

df['listed_in'] = df['listed_in'].str.split(', ')
df = df.explode('listed_in', ignore_index=True)
print(df[['country', 'listed_in']].head(10))

         country               listed_in
0  United States           Documentaries
1   South Africa  International TV Shows
2   South Africa               TV Dramas
3   South Africa            TV Mysteries
4         Unkown          Crime TV Shows
5         Unkown  International TV Shows
6         Unkown   TV Action & Adventure
7         Unkown              Docuseries
8         Unkown              Reality TV
9          India  International TV Shows


In [47]:
#listing all the ratings
print(df['rating'].unique())
df[df['rating'].str.contains('min')]
#we can see that the data shifted between ratings and duration

<StringArray>
[   'PG-13',    'TV-MA',       'PG',    'TV-14',    'TV-PG',     'TV-Y',
    'TV-Y7',        'R',     'TV-G',        'G',    'NC-17',   '74 min',
   '84 min',   '66 min',       'NR',        nan, 'TV-Y7-FV',       'UR']
Length: 18, dtype: str


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,duration_val,duration_unit,release_lag
14779,s5542,Movie,Louis C.K. 2017,Louis C.K.,Louis C.K.,United States,2017-04-04,2017,74 min,NaN,Movies,"Louis C.K. muses on religion, eternal love, gi...",2017,NaN,NaN,0
15409,s5795,Movie,Louis C.K.: Hilarious,Louis C.K.,Louis C.K.,United States,2016-09-16,2010,84 min,NaN,Movies,Emmy-winning comedy writer Louis C.K. brings h...,2016,NaN,NaN,6
15444,s5814,Movie,Louis C.K.: Live at the Comedy Store,Louis C.K.,Louis C.K.,United States,2016-08-15,2015,66 min,NaN,Movies,The comic puts his trademark hilarious/thought...,2016,NaN,NaN,1


In [48]:
#Filling missing values
df['rating'] = df['rating'].fillna("UR")

#migrating the misplaced durations,
df.loc[df['rating'].str.contains('min'), 'duration'] = df.loc[df['rating'].str.contains('min'), 'rating']

#Resetting the bad ratings
df.loc[df['rating'].str.contains('min'), 'rating'] = 'UR'
print(df['rating'].unique())

<StringArray>
[   'PG-13',    'TV-MA',       'PG',    'TV-14',    'TV-PG',     'TV-Y',
    'TV-Y7',        'R',     'TV-G',        'G',    'NC-17',       'UR',
       'NR', 'TV-Y7-FV']
Length: 14, dtype: str


In [49]:
#mapping the ratings
ratings_map = {
    'TV-G' : 'Kids',
    'G' : 'Kids',
    'TV-Y7-FV' : 'Kids',
    'TV-Y7' : 'Kids',
    'TV-Y' : 'Kids',
    'PG' : 'Kids',
    'TV-PG' : 'Kids',
    'PG-13' : 'Teens',
    'TV-14' : 'Teens',
    'TV-MA' : 'Adults',
    'R' : 'Adults',
    'NC-17' : 'Adults',
    'NR' : 'Adults',
    'UR' : 'Adults',
}
df['rating_categorie'] = df['rating'].map(ratings_map)
print(df['rating_categorie'].isna().sum())
df['rating_categorie'].value_counts()

0


rating_categorie
Adults    11421
Teens      7388
Kids       4945
Name: count, dtype: int64